# Qualidade de dados — BCI IV 2a (DuckDB)

Duplicatas, completude por campo e valores inválidos no Parquet de `data/processed/`.
Síntese e decisões no [`../README.md`](../README.md).

In [1]:
from pathlib import Path
import duckdb

data_dir = (Path("..") / "data" / "processed").as_posix()
tables = ["fact_trial", "dim_subject", "dim_class", "dim_session", "dim_run"]

con = duckdb.connect()
for table in tables:
    con.sql(f"CREATE VIEW {table} AS SELECT * FROM '{data_dir}/{table}.parquet'")

## Duplicatas

Chave primária de cada tabela e as duas chaves naturais da fato.

In [2]:
con.sql("""
    SELECT 'fact_trial.trial_id' AS chave, COUNT(*) AS linhas,
           COUNT(*) - COUNT(DISTINCT trial_id) AS duplicatas
    FROM fact_trial
    UNION ALL
    SELECT 'fact_trial (sujeito, sessão, run, trial_in_run)', COUNT(*),
           COUNT(*) - COUNT(DISTINCT (subject_id, session_id, run_id, trial_in_run))
    FROM fact_trial
    UNION ALL
    SELECT 'fact_trial (sujeito, sessão, run, onset_s)', COUNT(*),
           COUNT(*) - COUNT(DISTINCT (subject_id, session_id, run_id, onset_s))
    FROM fact_trial
    UNION ALL
    SELECT 'dim_subject.subject_id', COUNT(*), COUNT(*) - COUNT(DISTINCT subject_id) FROM dim_subject
    UNION ALL
    SELECT 'dim_class.class_id', COUNT(*), COUNT(*) - COUNT(DISTINCT class_id) FROM dim_class
    UNION ALL
    SELECT 'dim_session.session_id', COUNT(*), COUNT(*) - COUNT(DISTINCT session_id) FROM dim_session
    UNION ALL
    SELECT 'dim_run.run_id', COUNT(*), COUNT(*) - COUNT(DISTINCT run_id) FROM dim_run
""").df()

,chave,linhas,duplicatas
0,fact_trial.trial_id,5184,0
1,"fact_trial (sujeito, sessão, run, trial_in_run)",5184,0
2,"fact_trial (sujeito, sessão, run, onset_s)",5184,0
3,dim_subject.subject_id,9,0
4,dim_class.class_id,4,0
5,dim_session.session_id,2,0
6,dim_run.run_id,6,0


## Completude por campo

Percentual de valores não nulos em cada coluna de cada tabela.

In [3]:
con.sql(" UNION ALL ".join(
    f"""SELECT '{table}' AS tabela, column_name AS campo, column_type AS tipo,
               count AS linhas, 100 - null_percentage AS completude_pct
        FROM (SUMMARIZE {table})"""
    for table in tables
)).df()

,tabela,campo,tipo,linhas,completude_pct
0,fact_trial,trial_id,VARCHAR,5184,100.0
1,fact_trial,subject_id,VARCHAR,5184,100.0
2,fact_trial,session_id,VARCHAR,5184,100.0
3,fact_trial,run_id,VARCHAR,5184,100.0
4,fact_trial,class_id,SMALLINT,5184,100.0
5,fact_trial,trial_in_run,SMALLINT,5184,100.0
6,fact_trial,onset_s,DOUBLE,5184,100.0
7,fact_trial,duration_s,DOUBLE,5184,100.0
8,fact_trial,n_samples,INTEGER,5184,100.0
9,fact_trial,has_artifact,BOOLEAN,5184,100.0


## Valores inválidos

Uma linha por regra; `violacoes` = quantas linhas (ou grupos) a quebram.

In [4]:
con.sql("""
    WITH seq AS (
        SELECT onset_s - LAG(onset_s) OVER (
                   PARTITION BY subject_id, session_id, run_id ORDER BY trial_in_run) AS gap_s
        FROM fact_trial
    )
    SELECT 'class_id fora de 0–3' AS regra, COUNT(*) AS violacoes
    FROM fact_trial WHERE class_id NOT BETWEEN 0 AND 3
    UNION ALL
    SELECT 'trial_in_run fora de 0–47', COUNT(*)
    FROM fact_trial WHERE trial_in_run NOT BETWEEN 0 AND 47
    UNION ALL
    SELECT 'duration_s diferente de 4,0', COUNT(*) FROM fact_trial WHERE duration_s <> 4.0
    UNION ALL
    SELECT 'n_samples diferente de duration_s × 250', COUNT(*)
    FROM fact_trial WHERE n_samples <> duration_s * 250
    UNION ALL
    SELECT 'onset_s negativo', COUNT(*) FROM fact_trial WHERE onset_s < 0
    UNION ALL
    SELECT 'sentinela (-1 ou 999) em campo numérico', COUNT(*)
    FROM fact_trial
    WHERE class_id IN (-1, 999) OR trial_in_run IN (-1, 999) OR n_samples IN (-1, 999)
       OR onset_s IN (-1, 999) OR duration_s IN (-1, 999)
    UNION ALL
    SELECT 'subject_id fora do padrão A01–A09', COUNT(*)
    FROM fact_trial WHERE NOT regexp_full_match(subject_id, 'A0[1-9]')
    UNION ALL
    SELECT 'trial_id diferente da composição das chaves', COUNT(*)
    FROM fact_trial
    WHERE trial_id <> subject_id || '_' || session_id || '_' || run_id
                      || '_t' || lpad(trial_in_run::VARCHAR, 2, '0')
    UNION ALL
    SELECT 'subject_id sem linha em dim_subject', COUNT(*)
    FROM fact_trial ANTI JOIN dim_subject USING (subject_id)
    UNION ALL
    SELECT 'session_id sem linha em dim_session', COUNT(*)
    FROM fact_trial ANTI JOIN dim_session USING (session_id)
    UNION ALL
    SELECT 'run_id sem linha em dim_run', COUNT(*)
    FROM fact_trial ANTI JOIN dim_run USING (run_id)
    UNION ALL
    SELECT 'class_id sem linha em dim_class', COUNT(*)
    FROM fact_trial ANTI JOIN dim_class USING (class_id)
    UNION ALL
    SELECT 'run com nº de trials diferente de 48', COUNT(*)
    FROM (SELECT COUNT(*) AS n FROM fact_trial GROUP BY subject_id, session_id, run_id)
    WHERE n <> 48
    UNION ALL
    SELECT 'run × classe com nº de trials diferente de 12', COUNT(*)
    FROM (SELECT COUNT(*) AS n FROM fact_trial GROUP BY subject_id, session_id, run_id, class_id)
    WHERE n <> 12
    UNION ALL
    SELECT 'onset_s fora de ordem dentro do run', COUNT(*) FROM seq WHERE gap_s <= 0
    UNION ALL
    SELECT 'intervalo entre trials fora de 7,5–8,5 s', COUNT(*)
    FROM seq WHERE gap_s NOT BETWEEN 7.5 AND 8.5
    UNION ALL
    SELECT 'dim_subject.age fora de 10–100', COUNT(*)
    FROM dim_subject WHERE age NOT BETWEEN 10 AND 100
    UNION ALL
    SELECT 'dim_subject.sex fora de {M, F}', COUNT(*)
    FROM dim_subject WHERE sex NOT IN ('M', 'F')
""").df()

,regra,violacoes
0,class_id fora de 0–3,0
1,trial_in_run fora de 0–47,0
2,"duration_s diferente de 4,0",0
3,n_samples diferente de duration_s × 250,0
4,onset_s negativo,0
5,sentinela (-1 ou 999) em campo numérico,0
6,subject_id fora do padrão A01–A09,0
7,trial_id diferente da composição das chaves,0
8,subject_id sem linha em dim_subject,0
9,session_id sem linha em dim_session,0


## Trials marcados como artefato (quarentena)

Marcados na fonte; ficam na fato com `has_artifact = TRUE`.

In [5]:
con.sql("""
    SELECT COALESCE(subject_id, 'total') AS subject_id,
           COUNT(*) FILTER (has_artifact) AS n_artefato,
           ROUND(100.0 * COUNT(*) FILTER (has_artifact) / COUNT(*), 1) AS pct_artefato
    FROM fact_trial
    GROUP BY ROLLUP (subject_id)
    ORDER BY GROUPING(subject_id), subject_id
""").df()

,subject_id,n_artefato,pct_artefato
0,A01,22,3.8
1,A02,23,4.0
2,A03,33,5.7
3,A04,86,14.9
4,A05,38,6.6
5,A06,142,24.7
6,A07,28,4.9
7,A08,41,7.1
8,A09,75,13.0
9,total,488,9.4


In [6]:
# trials restantes por sujeito × sessão × classe (72 no original)
con.sql("""
    SELECT c.class_name,
           SUM(n_artefato)::INT AS n_artefato,
           MIN(restantes) AS min_restantes, MAX(restantes) AS max_restantes
    FROM (
        SELECT subject_id, session_id, class_id,
               COUNT(*) FILTER (has_artifact) AS n_artefato,
               COUNT(*) FILTER (NOT has_artifact) AS restantes
        FROM fact_trial
        GROUP BY ALL
    )
    JOIN dim_class c USING (class_id)
    GROUP BY 1
    ORDER BY 1
""").df()

,class_name,n_artefato,min_restantes,max_restantes
0,feet,122,49,72
1,left_hand,131,53,71
2,right_hand,115,55,71
3,tongue,120,53,72
